# Notebook for measuring runtime of HYBRID: Hashing, bucketing and similarity value computation 

In [ ]:
import os
import sys
import itertools


def find_project_root(target_folder="masteroppgave"):
    """Find the absolute path of a folder by searching upward."""
    currentdir = os.path.abspath("__file__")  # Get absolute script path
    while True:
        if os.path.basename(currentdir) == target_folder:
            return currentdir  # Found the target folder
        parentdir = os.path.dirname(currentdir)
        if parentdir == currentdir:  # Stop at filesystem root
            return None
        currentdir = parentdir  # Move one level up

# Example usage
project_root = find_project_root("masteroppgave")

if project_root:
    sys.path.append(project_root)
    print(f"Project root found: {project_root}")
else:
    raise RuntimeError("Could not find 'masteroppgave' directory")

from utils.helpers.measure_similarities import *


# Disk

In [2]:
MEASURE="disk_dtw_cy"
CITY="rome"
DATA_SIZE = 50  # Example dataset sizes

#Parameters
DIAMETER_BUCKETING_LIST = [0.5]  # Diameter to use for bucketing
LAYERS_BUCKETING_LIST = [2, 3]  # Number of layers to use for bucketing
DISKS_BUCKETING_LIST = [20]  # Number of disks to use for bucketing

DIAMETER_COMPRESSION_LIST = [1.5]  # Diameter to use for trajectory compression
LAYERS_COMPRESSION_LIST = [2]  # Number of layers to use for trajectory compression
DISKS_COMPRESSION_LIST = [40]  # Number of disks to use for trajectory compression

#Strategies
BUCKETING_METHOD = "loose"

# Logistics
PARALLEL_JOBS = 8
ITERATIONS = 1

In [3]:
if "disk_dtw_cy" or "disk_frechet_cy" in MEASURE:
    scheme = "disk"
elif "grid_dtw_cy" or "grid_frechet_cy" in MEASURE:
    scheme = "grid"


if "dtw" in MEASURE:
    measure = "dtw"
elif "frechet" in MEASURE:
    measure = "frechet"

file_name = f"runtimes_{CITY}_disk_hybrid_{measure}_db({min(DIAMETER_BUCKETING_LIST)}-{max(DIAMETER_BUCKETING_LIST)})_dc({min(DIAMETER_COMPRESSION_LIST)}-{max(DIAMETER_COMPRESSION_LIST)})_lb({min(LAYERS_BUCKETING_LIST)}-{max(LAYERS_BUCKETING_LIST)})_lc({min(LAYERS_COMPRESSION_LIST)}-{max(LAYERS_COMPRESSION_LIST)})_ndb({min(DISKS_BUCKETING_LIST)}-{max(DISKS_BUCKETING_LIST)})_ndc({min(DISKS_COMPRESSION_LIST)}-{max(DISKS_COMPRESSION_LIST)})_ds({DATA_SIZE})_bm({BUCKETING_METHOD}).csv"

output_path = f"../../../results_hashed/runtimes/hybrid/{CITY}/{BUCKETING_METHOD}/{file_name}"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

In [ ]:
param_combinations = list(itertools.product(DIAMETER_BUCKETING_LIST, LAYERS_BUCKETING_LIST, DISKS_BUCKETING_LIST, DIAMETER_COMPRESSION_LIST, LAYERS_COMPRESSION_LIST, DISKS_COMPRESSION_LIST))

first_write = True

print(f" Current param config: \n \tHYBRID: YES \n\tBUCKETING_METHOD: {BUCKETING_METHOD}\n\tCITY: {CITY}\n\tMEASURE: {measure}\n\tDATA_SIZE: {DATA_SIZE}\n\tSCHEME: {scheme} \n\tPARALLEL_JOBS: {PARALLEL_JOBS} \n\tITERATIONS: {ITERATIONS} \n\n")

# Iterate over each combination and run the function
for diameter_bucketing, layers_bucketing, disks_bucketing, diameter_compression, layers_compression, disks_compression in param_combinations:
    print(f"Running with diameter_bucketing={diameter_bucketing}, layers_bucketing={layers_bucketing}, disks_bucketing={disks_bucketing}, diameter_compression={diameter_compression}, layers_compression={layers_compression}, disks_compression={disks_compression}")
    
    # Call the function with the current combination of parameters
    df_result = compute_hashed_similarity_runtimes_with_bucketing_hybrid(
        measure=MEASURE,
        city=CITY, 
        diameter_bucketing=diameter_bucketing, 
        layers_bucketing=layers_bucketing, 
        disks_bucketing=disks_bucketing, 
        diameter_compression=diameter_compression, 
        layers_compression=layers_compression, 
        disks_compression=disks_compression, 
        parallel_jobs=PARALLEL_JOBS, 
        data_size = DATA_SIZE, 
        iterations=ITERATIONS, 
        bucketing_method=BUCKETING_METHOD
    )

    # Add parameters to result DataFrame
    df_result["City"] = CITY
    df_result["Measure"] = measure
    df_result["Diameter_bucketing"] = diameter_bucketing
    df_result["Layers_bucketing"] = layers_bucketing
    df_result["Disks_bucketing"] = disks_bucketing
    df_result["Diameter_compression"] = diameter_compression
    df_result["Layers_compression"] = layers_compression
    df_result["Disks_compression"] = disks_compression
    df_result["Size"] = DATA_SIZE

    desired_order = [
        "City", "Measure", "Diameter_bucketing", "Layers_bucketing", "Disks_bucketing",
        "Diameter_compression", "Layers_compression", "Disks_compression", "Size", 
        "Average Similarity Computation Time (Seconds)",
        "Average Hash Generation Time (Seconds) - Bucketing",
        "Average Hash Generation Time (Seconds) - Compression",
        "Average Bucket Distribution Time (Seconds)",
        "Total time (Seconds)",
    ]

    # Reorder columns
    df_result = df_result[desired_order]

    # Save the DataFrame to a CSV file
    df_result.to_csv(output_path, mode='a', header=first_write, index=False)
    first_write = False
    


## Grid

In [5]:
MEASURE="grid_dtw_cy"
CITY="rome"
DATA_SIZE = 50  # Example dataset sizes

# Parameters
RESOLUTION_BUCKETING_LIST = [0.5] # Resolution to use for bucketing
LAYERS_BUCKETING_LIST = [2, 3]  # Number of layers to use for bucketing

RESOLUTION_COMPRESSION_LIST = [0.5]  # Resolution to use for trajectory compression
LAYERS_COMPRESSION_LIST = [2]  # Number of layers to use for trajectory compression

#Strategies
BUCKETING_METHOD = "loose"

# Logistics
PARALLEL_JOBS = 8
ITERATIONS = 1

In [6]:
if "disk_dtw_cy" or "disk_frechet_cy" in MEASURE:
    scheme = "disk"
elif "grid_dtw_cy" or "grid_frechet_cy" in MEASURE:
    scheme = "grid"

if "dtw" in MEASURE:
    measure = "dtw"
elif "frechet" in MEASURE:
    measure = "frechet"

file_name = f"runtimes_{CITY}_grid_hybrid_{measure}_rb({min(RESOLUTION_BUCKETING_LIST)}-{max(RESOLUTION_BUCKETING_LIST)})_rc({min(RESOLUTION_COMPRESSION_LIST)}-{max(RESOLUTION_COMPRESSION_LIST)})_lb({min(LAYERS_BUCKETING_LIST)}-{max(LAYERS_BUCKETING_LIST)})_lc({min(LAYERS_COMPRESSION_LIST)}-{max(LAYERS_COMPRESSION_LIST)})_ds({DATA_SIZE})_bm({BUCKETING_METHOD}).csv"

output_path = f"../../../results_hashed/runtimes/hybrid/{CITY}/{BUCKETING_METHOD}/{file_name}"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

In [ ]:
param_combinations = list(itertools.product(RESOLUTION_BUCKETING_LIST, LAYERS_BUCKETING_LIST, RESOLUTION_COMPRESSION_LIST, LAYERS_COMPRESSION_LIST))

first_write = True

print(f" Current param config: \n\tHYBRID: YES \n\tBUCKETING_METHOD: {BUCKETING_METHOD}\n\tCITY: {CITY}\n\tMEASURE: {measure}\n\tDATA_SIZE: {DATA_SIZE}\n\tSCHEME: {scheme} \n\tPARALLEL_JOBS: {PARALLEL_JOBS} \n\tITERATIONS: {ITERATIONS} \n\n")


# Iterate over each combination and run the function
for resolution_bucketing, layers_bucketing, resolution_compression, layers_compression in param_combinations:
    print(f"Running with resolution_bucketing={resolution_bucketing}, layers_bucketing={layers_bucketing}, resolution_compression={resolution_compression}, layers_compression={layers_compression}")

    df_result = compute_hashed_similarity_runtimes_with_bucketing_hybrid(
        measure=MEASURE,
        city=CITY,
        res_bucketing=resolution_bucketing,
        layers_bucketing=layers_bucketing,
        res_compression=resolution_compression,
        layers_compression=layers_compression,
        parallel_jobs=PARALLEL_JOBS,
        data_size=DATA_SIZE,
        iterations=ITERATIONS,
        bucketing_method=BUCKETING_METHOD
    )

    # Add parameters to result DataFrame
    df_result["City"] = CITY
    df_result["Measure"] = measure
    df_result["Resolution_bucketing"] = resolution_bucketing
    df_result["Layers_bucketing"] = layers_bucketing
    df_result["Resolution_compression"] = resolution_compression
    df_result["Layers_compression"] = layers_compression
    df_result["Size"] = DATA_SIZE

    desired_order = [
        "City", "Measure", "Resolution_bucketing", "Layers_bucketing",
        "Resolution_compression", "Layers_compression", "Size",
        "Average Similarity Computation Time (Seconds)",
        "Average Hash Generation Time (Seconds) - Bucketing",
        "Average Hash Generation Time (Seconds) - Compression",
        "Average Bucket Distribution Time (Seconds)",
        "Total time (Seconds)",
    ]

    # Reorder columns
    df_result = df_result[desired_order]

    # Save the DataFrame to a CSV file
    df_result.to_csv(output_path, mode='a', header=first_write, index=False)
    first_write = False
    
